In [2]:
#packages
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import seaborn as sns
import os
from matplotlib import colormaps
import glob
import matplotlib.patches as mpatches
import pingouin as pg
from scipy.stats import norm, normaltest
from scipy.stats import sem

from scipy.stats import wilcoxon  # Pour test apparié
from scipy.stats import mannwhitneyu  # Pour test indépendant
from scipy.stats import ttest_rel
from scipy.stats import shapiro

In [3]:
#Import all the data

#to compile files from one folder
folder_path = "../../Logs"

# List all the jsons from the folder
json_files = glob.glob(os.path.join(folder_path, "*.json"))

# Initialise a list to stock dataframes
df_list = []

# Read and exctract all data
for file in json_files:
    with open(file, 'r') as f:
        data = json.load(f)
        #df_list.append(pd.DataFrame(data['steps'])) 
        # Extract ppt number from the title of the file 
        participant_id = os.path.basename(file).split("_")[1]  # suppose that the file name is Logs_XXXX-XX-XX.json

        # Create df and add participant column
        df_temp = pd.DataFrame(data['steps'])
        df_temp["participant"] = participant_id  

        # add to the df list
        df_list.append(df_temp)

# Concat all dataframes
df_base = pd.concat(df_list, ignore_index=True)
print(df_base.head())
print(df_base.shape)

   lightIndex   lightTime  toyIndex     toyTime  soundIndex   soundTime  \
0           0  402.638947         7  402.647339           1  402.639038   
1           0  408.350403         7  408.361542           7  408.350464   
2           0  415.379303         7  415.390381           7  415.379364   
3           0  422.183411         1  422.194214           1  422.183472   
4           0  427.830566         1  427.841797           1  427.830627   

   responseTime  correct  distractorIndex1  distractorIndex2  \
0      1.693268     True                 5                -1   
1      0.000000    False                 3                -1   
2      2.783691     True                 3                -1   
3      1.627960     True                 3                 5   
4      1.715912     True                -1                -1   

   distractorIndex3 participant  
0                -1     18.json  
1                -1     18.json  
2                -1     18.json  
3                 7     18.j

In [4]:
#function to organise the dataframe

def process_steps(df_base, sound_index_col='soundIndex', toy_index_col='toyIndex'):
    
    # Add colomn 'cue' 
    df_base['cue'] = np.where(df_base['soundIndex'] == df_base['toyIndex'], 'Valid', 'Invalid')

    # Add colomn 'position'
    df_base['position'] = np.where(df_base[toy_index_col].isin([1, 7]), 'Rear',
                              np.where(df_base[toy_index_col].isin([3, 5]), 'Front', 'autre'))

    # delet un-used colomns
    df_base.drop(columns=['lightIndex', 'lightTime'], errors='ignore', inplace=True)

    #add participant columns
    #if 'participant' not in df_base.columns:
     #   df_base['participant'] = participant_num  # Ajouter la colonne 'participant' uniquement si elle n'existe pas

    #add localisation
    df_base['lateralisation'] = np.where(df_base[toy_index_col].isin([1, 3]), 'Left',
                              np.where(df_base[toy_index_col].isin([7, 5]), 'Right', 'autre'))
    
    #add distractor condition (0, 1, 3)
    df_base['distractor'] = np.sum(df_base[['distractorIndex1', 'distractorIndex2', 'distractorIndex3']].ne(-1), axis=1)
   

    #add distractor modality (Near, Far)
    df_base['distractor_unique'] = df_base[['distractorIndex1', 'distractorIndex2', 'distractorIndex3']].\
        apply(lambda row: next((x for x in row if x != -1), np.nan), axis=1)

        # "near"
    near_conditions = (
        ((df_base['toyIndex'] == 1) & (df_base['distractor_unique'] == 3)) |
        ((df_base['toyIndex'] == 3) & (df_base['distractor_unique'] == 5)) |
        ((df_base['toyIndex'] == 5) & (df_base['distractor_unique'] == 3)) |
        ((df_base['toyIndex'] == 7) & (df_base['distractor_unique'] == 5))
    )
    
        # "far"
    far_conditions = (
        ((df_base['toyIndex'] == 1) & (df_base['distractor_unique'] == 5)) |
        ((df_base['toyIndex'] == 3) & (df_base['distractor_unique'] == 7)) |
        ((df_base['toyIndex'] == 5) & (df_base['distractor_unique'] == 1)) |
        ((df_base['toyIndex'] == 7) & (df_base['distractor_unique'] == 3))
    )
    
        # Apply
    df_base['moda_dist'] = np.where(df_base['distractor'] == 1, 
                                         np.where(near_conditions, 'Near', 
                                                  np.where(far_conditions, 'Far', 'None')), 
                                         'DA') #DA = don't apply
    
        # drop column
    df_base.drop(columns=['distractor_unique'], inplace=True)

  
    df_base['distractor'] = np.where(
    df_base['distractor'] == 0, 'Zero',
    np.where(
        df_base['distractor'] == 3, 'Three',
        np.where(
            df_base['distractor'] == 1,
            df_base['moda_dist'].apply(lambda x: f"One_{x}" if x != 'DA' else 'DA'),
            df_base['distractor']
        )
    )
)
    return df_base
df_step = process_steps(df_base)

In [5]:
#cleaning
df_stepclean = df_step[df_step['responseTime'] > 0.100] # under 100 ms RT we consider it a misclic because it has been reported as imposible to respond under 100 ms

# Remove Rear trials with responseTime < 400 ms
#df_stepclean = df_step[~((df_step['position'] == 'Rear') & (df_step['responseTime'] < 0.400))]

# Verify the filtering
print("Original data shape:", df_step.shape)
print("Filtered data shape:", df_stepclean.shape)
#print("\nRemaining Rear trials under 400 ms:", 
#      df_stepclean[(df_stepclean['position'] == 'Rear') & (df_stepclean['responseTime'] < 0.400)].shape[0])

df_stepclean


Original data shape: (3200, 15)
Filtered data shape: (3074, 15)


,toyIndex,toyTime,soundIndex,soundTime,responseTime,correct,distractorIndex1,distractorIndex2,distractorIndex3,participant,cue,position,lateralisation,distractor,moda_dist
0,7,402.647339,1,402.639038,1.693268,True,5,-1,-1,18.json,Invalid,Rear,Right,One_Near,Near
2,7,415.390381,7,415.379364,2.783691,True,3,-1,-1,18.json,Valid,Rear,Right,One_Far,Far
3,1,422.194214,1,422.183472,1.627960,True,3,5,7,18.json,Valid,Rear,Left,Three,DA
4,1,427.841797,1,427.830627,1.715912,True,-1,-1,-1,18.json,Valid,Rear,Left,Zero,DA
5,3,433.577759,3,433.566162,1.293121,True,7,-1,-1,18.json,Valid,Front,Left,One_Far,Far
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,1,1808.725952,1,1808.714722,1.451172,True,3,-1,-1,9.json,Valid,Rear,Left,One_Near,Near
3196,1,1814.194702,7,1814.183594,2.597290,True,3,-1,-1,9.json,Invalid,Rear,Left,One_Near,Near
3197,7,1820.810913,7,1820.799561,1.314331,True,1,3,5,9.json,Valid,Rear,Right,Three,DA
3198,7,1826.146240,7,1826.136841,1.258789,True,5,-1,-1,9.json,Valid,Rear,Right,One_Near,Near


In [ ]:
## keep only success trials 
df_success_trials = df_stepclean[df_stepclean["correct"] == True]
df_success_trials = df_success_trials.groupby(['participant', 'cue', 'position', 'distractor', 'correct'])['responseTime'].mean().reset_index()
print(df_success_trials)
#export the full datafram : does not contain accuracy nore false trials
df_success_trials.to_excel("1.2.R_analysis/dataframes/df_success_trials.xlsx", index=False)
print("done")

    participant      cue position distractor  correct  responseTime
0        1.json  Invalid    Front    One_Far     True      0.915741
1        1.json  Invalid    Front   One_Near     True      0.839233
2        1.json  Invalid    Front      Three     True      1.091019
3        1.json  Invalid    Front       Zero     True      0.840088
4        1.json  Invalid     Rear    One_Far     True      2.374268
..          ...      ...      ...        ...      ...           ...
315      9.json    Valid    Front       Zero     True      0.760311
316      9.json    Valid     Rear    One_Far     True      1.595959
317      9.json    Valid     Rear   One_Near     True      1.632982
318      9.json    Valid     Rear      Three     True      1.510213
319      9.json    Valid     Rear       Zero     True      1.471878

[320 rows x 6 columns]
done


In [7]:
#we want to create a dataframe for accuracy rates
#we take from df_base, before the filtering of wrong trials

import pandas as pd

# Group by participant AND the other factors
grouped = df_base.groupby(['participant', 'cue', 'position', 'distractor'])

# Compute correct rate and average response time per group
accuracy_per_participant = grouped.agg(
    accuracy=('correct', 'mean'),
    response_time=('responseTime', 'mean')
).reset_index()
accuracy_per_participant

,participant,cue,position,distractor,accuracy,response_time
0,1.json,Invalid,Front,One_Far,1.0000,0.915741
1,1.json,Invalid,Front,One_Near,1.0000,0.839233
2,1.json,Invalid,Front,Three,1.0000,1.091019
3,1.json,Invalid,Front,Zero,1.0000,0.840088
4,1.json,Invalid,Rear,One_Far,1.0000,2.374268
...,...,...,...,...,...,...
315,9.json,Valid,Front,Zero,1.0000,0.760311
316,9.json,Valid,Rear,One_Far,0.9375,1.579762
317,9.json,Valid,Rear,One_Near,1.0000,1.632982
318,9.json,Valid,Rear,Three,0.9375,1.415825


In [ ]:
accuracy_per_participant.to_excel("1.2.R_analysis/dataframes/df_accuracy.xlsx", index=False)

In [9]:
mapping = {
    "One_Far": "Absente",
    "Zero": "Absente",
    "One_Near": "Presente",
    "Three": "Presente"
}

df_accuracy_reactegorized = accuracy_per_participant.copy()
df_accuracy_reactegorized["distractor1"] = df_accuracy_reactegorized["distractor"].map(mapping)


In [ ]:
df_accuracy_reactegorized.to_excel("1.2.R_analysis/dataframes/df_recat_acc.xlsx", index=False)

In [11]:
accuracy_per_participant

,participant,cue,position,distractor,accuracy,response_time
0,1.json,Invalid,Front,One_Far,1.0000,0.915741
1,1.json,Invalid,Front,One_Near,1.0000,0.839233
2,1.json,Invalid,Front,Three,1.0000,1.091019
3,1.json,Invalid,Front,Zero,1.0000,0.840088
4,1.json,Invalid,Rear,One_Far,1.0000,2.374268
...,...,...,...,...,...,...
315,9.json,Valid,Front,Zero,1.0000,0.760311
316,9.json,Valid,Rear,One_Far,0.9375,1.579762
317,9.json,Valid,Rear,One_Near,1.0000,1.632982
318,9.json,Valid,Rear,Three,0.9375,1.415825


In [ ]:
#Before filtering 
# count true and false per participant
summary = df_base.groupby('participant')['correct'].value_counts().unstack(fill_value=0)
# Renames columns
summary.columns = ['Incorrect (False)', 'Correct (True)']
summary['%correct'] = (summary['Correct (True)'] / (summary['Correct (True)'] + summary['Incorrect (False)'])) * 100
#print(summary)

#overall hit rates
overall_percent_correct = df_base['correct'].mean() * 100
print("Overall % Correct:", overall_percent_correct)
std_across_participants = summary['%correct'].std()
print("Std of % Correct across participants:", std_across_participants)


Overall % Correct: 93.75
Std of % Correct across participants: 3.4352564448822833


In [23]:
#After filtering
# count true and false per participant
summary = df_stepclean.groupby('participant')['correct'].value_counts().unstack(fill_value=0)
# Renames columns
summary.columns = ['Incorrect (False)', 'Correct (True)']
summary['%correct'] = (summary['Correct (True)'] / (summary['Correct (True)'] + summary['Incorrect (False)'])) * 100
print(summary)

#overall hit rates
overall_percent_correct = df_stepclean['correct'].mean() * 100
print("Mean % Correct:", overall_percent_correct)
std_across_participants = summary['%correct'].std()
print("Std of % Correct across participants:", std_across_participants)
total_incorrect = summary['Incorrect (False)'].sum()
print("Total trials Incorrect:", total_incorrect)
total_correct = summary['Correct (True)'].sum()
print("Total trials correct:", total_correct)

             Incorrect (False)  Correct (True)   %correct
participant                                              
1.json                       2             157  98.742138
11.json                      3             154  98.089172
12.json                      3             156  98.113208
13.json                      9             143  94.078947
14.json                      2             151  98.692810
16.json                      3             155  98.101266
17.json                      5             153  96.835443
18.json                      7             147  95.454545
2.json                       4             154  97.468354
20.json                      1             140  99.290780
21.json                      2             150  98.684211
22.json                      2             151  98.692810
24.json                      5             143  96.621622
25.json                      1             150  99.337748
4.json                       1             156  99.363057
5.json        